In [ ]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
project_root = Path("..")
cleaned_dir = project_root / "data" / "cleaned"
raw_dir = project_root / "data" / "raw"

con = duckdb.connect()

fuel_cost_per_mile = 0.20

In [ ]:
# load saved results from notebook 04

zone_hour_results = pd.read_csv(cleaned_dir / "zone_hour_results.csv")
zone_hour_stats = pd.read_csv(cleaned_dir / "zone_hour_stats.csv")
zone_hour_flows = pd.read_csv(cleaned_dir / "zone_hour_flows.csv")
zone_hour_transitions = pd.read_csv(cleaned_dir / "zone_hour_transitions.csv")

In [ ]:
# rebuild valid zones

valid_zones = sorted(zone_hour_results["PULocationID"].unique())

In [ ]:
# build a value lookup from notebook 04 results

value_lookup = {}
for _, row in zone_hour_results.iterrows():
    value_lookup[(int(row["PULocationID"]), int(row["start_hour"]))] = float(
        row["expected_net_earnings_per_hour"]
    )

In [ ]:
# build candidate top zones by hour

top_k = 5

top_zones_by_hour = {}

for hour in range(24):
    temp = (
        zone_hour_results[zone_hour_results["start_hour"] == hour]
        .sort_values("expected_net_earnings_per_hour", ascending=False)
        .head(top_k)
    )
    top_zones_by_hour[hour] = temp["PULocationID"].astype(int).tolist()


In [ ]:
# build reposition travel sets
# want avg travel time and distance from zone A to zone B by hour,
# using observed trips as a proxy

reposition_stats = con.execute(f"""
SELECT
    PULocationID,
    DOLocationID,
    EXTRACT(hour FROM tpep_pickup_datetime) AS pickup_hour,
    COUNT(*) AS trips,
    AVG(trip_distance) AS avg_reposition_miles,
    AVG(
        EXTRACT(EPOCH FROM (tpep_dropoff_datetime - tpep_pickup_datetime)) / 60.0
    ) AS avg_reposition_minutes
FROM read_parquet('{cleaned_dir}/yellow_tripdata_2024-*.parquet')
WHERE
    total_amount > 0
    AND PULocationID BETWEEN 1 AND 263
    AND DOLocationID BETWEEN 1 AND 263
GROUP BY
    PULocationID,
    DOLocationID,
    pickup_hour
HAVING COUNT(*) >= 30
""").fetchdf()

# filter to valid zones

reposition_stats = reposition_stats[
    reposition_stats["PULocationID"].isin(valid_zones)
    & reposition_stats["DOLocationID"].isin(valid_zones)
].copy()

# build a lookup

reposition_lookup = {}

for _, row in reposition_stats.iterrows():
    reposition_lookup[
        (int(row["PULocationID"]), int(row["DOLocationID"]), int(row["pickup_hour"]))
    ] = {
        "minutes": float(row["avg_reposition_minutes"]),
        "miles": float(row["avg_reposition_miles"]),
    }

In [ ]:
# rebuild existing simulation lookups,
# since we still need the normal trip process after repositioning

# stats lookup

stats_lookup = {}
for _, row in zone_hour_stats.iterrows():
    stats_lookup[(int(row["PULocationID"]), int(row["pickup_hour"]))] = {
        "avg_net_trip_earnings": float(row["avg_net_trip_earnings"]),
        "avg_trip_minutes": float(row["avg_trip_minutes"]),
        "zone_name": row["Zone"],
        "borough": row["Borough"],
    }

# wait lookup

wait_lookup = {}
for _, row in zone_hour_flows.iterrows():
    wait_lookup[(int(row["zone_id"]), int(row["hour"]))] = float(row["expected_wait_minutes"])

# transitions lookup

transitions_lookup = {}
for (pu, hr), group in zone_hour_transitions.groupby(["PULocationID", "pickup_hour"]):
    destinations = group["DOLocationID"].astype(int).to_numpy()
    probabilities = group["transition_prob"].to_numpy(dtype=float)
    probabilities = probabilities / probabilities.sum()

    transitions_lookup[(int(pu), int(hr))] = {
        "destinations": destinations,
        "probabilities": probabilities,
    }

In [ ]:
# decision rule: should driver reposition?

# compare current zone value, candidate zone value
# minus reposition fuel cost, minus the lost time traveling there

# lost time is an earnings penalty using the target zone's expected hourly value

def choose_reposition_zone(current_zone, current_hour):
    current_value = value_lookup.get((current_zone, current_hour), np.nan)

    if np.isnan(current_value):
        return None

    best_zone = current_zone
    best_gain = 0.0

    candidate_zones = top_zones_by_hour.get(current_hour, [])

    for candidate_zone in candidate_zones:
        if candidate_zone == current_zone:
            continue

        candidate_value = value_lookup.get((candidate_zone, current_hour), np.nan)
        if np.isnan(candidate_value):
            continue

        reposition_key = (current_zone, candidate_zone, current_hour)
        if reposition_key not in reposition_lookup:
            continue

        reposition_minutes = reposition_lookup[reposition_key]["minutes"]
        reposition_miles = reposition_lookup[reposition_key]["miles"]

        fuel_cost = fuel_cost_per_mile * reposition_miles
        time_cost = candidate_value * (reposition_minutes / 60.0)

        gain = candidate_value - current_value - fuel_cost - time_cost

        if gain > best_gain:
            best_gain = gain
            best_zone = candidate_zone

    if best_zone == current_zone:
        return None

    return {
        "new_zone": best_zone,
        "gain": best_gain,
        "minutes": reposition_lookup[(current_zone, best_zone, current_hour)]["minutes"],
        "miles": reposition_lookup[(current_zone, best_zone, current_hour)]["miles"],
    }

In [ ]:
# new simulation with optional repositioning

def simulate_shift_with_reposition(start_zone, start_hour, shift_minutes=480, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    current_zone = int(start_zone)
    elapsed_minutes = 0
    total_net_earnings = 0
    trip_count = 0
    reposition_count = 0

    initial_hour = start_hour % 24
    initial_wait = wait_lookup.get((current_zone, initial_hour), 5)
    elapsed_minutes += initial_wait

    while elapsed_minutes < shift_minutes:
        current_hour = (start_hour + int(elapsed_minutes / 60)) % 24

        # optional reposition decision before waiting for next trip
        reposition_decision = choose_reposition_zone(current_zone, current_hour)

        if reposition_decision is not None:
            reposition_minutes = reposition_decision["minutes"]
            reposition_miles = reposition_decision["miles"]

            if elapsed_minutes + reposition_minutes >= shift_minutes:
                break

            elapsed_minutes += reposition_minutes
            total_net_earnings -= fuel_cost_per_mile * reposition_miles
            current_zone = reposition_decision["new_zone"]
            reposition_count += 1

            current_hour = (start_hour + int(elapsed_minutes / 60)) % 24

        stats_key = (current_zone, current_hour)
        trans_key = (current_zone, current_hour)

        if stats_key not in stats_lookup or trans_key not in transitions_lookup:
            break

        trip_stats = stats_lookup[stats_key]
        trip_minutes = trip_stats["avg_trip_minutes"]
        wait_minutes = wait_lookup.get((current_zone, current_hour), 5)

        if elapsed_minutes + trip_minutes + wait_minutes > shift_minutes:
            break

        total_net_earnings += trip_stats["avg_net_trip_earnings"]
        elapsed_minutes += trip_minutes + wait_minutes
        trip_count += 1

        destinations = transitions_lookup[trans_key]["destinations"]
        probabilities = transitions_lookup[trans_key]["probabilities"]

        current_zone = int(rng.choice(destinations, p=probabilities))

    hours_worked = elapsed_minutes / 60

    if hours_worked <= 0 or trip_count == 0:
        return {
            "net_earnings_per_hour": np.nan,
            "trip_count": 0,
            "reposition_count": reposition_count
        }

    return {
        "net_earnings_per_hour": total_net_earnings / hours_worked,
        "trip_count": trip_count,
        "reposition_count": reposition_count
    }

In [ ]:
# baseline simulation without repositioning
# for comparison

def simulate_shift_baseline(start_zone, start_hour, shift_minutes=480, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    current_zone = int(start_zone)
    elapsed_minutes = 0
    total_net_earnings = 0
    trip_count = 0

    initial_hour = start_hour % 24
    initial_wait = wait_lookup.get((current_zone, initial_hour), 5)
    elapsed_minutes += initial_wait

    while elapsed_minutes < shift_minutes:
        current_hour = (start_hour + int(elapsed_minutes / 60)) % 24

        stats_key = (current_zone, current_hour)
        trans_key = (current_zone, current_hour)

        if stats_key not in stats_lookup or trans_key not in transitions_lookup:
            break

        trip_stats = stats_lookup[stats_key]
        trip_minutes = trip_stats["avg_trip_minutes"]
        wait_minutes = wait_lookup.get((current_zone, current_hour), 5)

        if elapsed_minutes + trip_minutes + wait_minutes > shift_minutes:
            break

        total_net_earnings += trip_stats["avg_net_trip_earnings"]
        elapsed_minutes += trip_minutes + wait_minutes
        trip_count += 1

        destinations = transitions_lookup[trans_key]["destinations"]
        probabilities = transitions_lookup[trans_key]["probabilities"]

        current_zone = int(rng.choice(destinations, p=probabilities))

    hours_worked = elapsed_minutes / 60

    if hours_worked <= 0 or trip_count == 0:
        return {
            "net_earnings_per_hour": np.nan,
            "trip_count": 0
        }

    return {
        "net_earnings_per_hour": total_net_earnings / hours_worked,
        "trip_count": trip_count
    }

In [ ]:
# evaluate function

def evaluate_strategy(start_zone, start_hour, n_simulations=100, reposition=False, seed=123):
    earnings_results = []
    trip_results = []
    reposition_results = []

    rng = np.random.default_rng(seed)

    for _ in range(n_simulations):
        if reposition:
            sim = simulate_shift_with_reposition(
                start_zone=start_zone,
                start_hour=start_hour,
                rng=rng
            )
            reposition_results.append(sim["reposition_count"])
        else:
            sim = simulate_shift_baseline(
                start_zone=start_zone,
                start_hour=start_hour,
                rng=rng
            )

        earnings_results.append(sim["net_earnings_per_hour"])
        trip_results.append(sim["trip_count"])

    out = {
        "mean_eph": np.nanmean(earnings_results),
        "std_eph": np.nanstd(earnings_results),
        "mean_trip_count": np.nanmean(trip_results),
    }

    if reposition:
        out["mean_reposition_count"] = np.nanmean(reposition_results)

    return out

In [ ]:
# compare baseline vs repositioning
# once this works, run it for all zones

comparison_results = []

for start_hour in range(24):

    print("Start hour:", start_hour)
    for zone_id in valid_zones:   
        base = evaluate_strategy(
            start_zone=zone_id,
            start_hour=start_hour,
            n_simulations=50,
            reposition=False,
            seed=1000 + start_hour * 100 + zone_id
        )

        repo = evaluate_strategy(
            start_zone=zone_id,
            start_hour=start_hour,
            n_simulations=50,
            reposition=True,
            seed=2000 + start_hour * 100 + zone_id
        )

        comparison_results.append({
            "PULocationID": zone_id,
            "start_hour": start_hour,
            "baseline_eph": base["mean_eph"],
            "reposition_eph": repo["mean_eph"],
            "gain_from_reposition": repo["mean_eph"] - base["mean_eph"],
            "avg_repositions": repo["mean_reposition_count"]
        })

comparison_results = pd.DataFrame(comparison_results)
comparison_results.sort_values("gain_from_reposition", ascending=False).head(10)


In [ ]:
# useful first plot

hour_gain = (
    comparison_results
    .groupby("start_hour")["gain_from_reposition"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(8,5))
plt.plot(hour_gain["start_hour"], hour_gain["gain_from_reposition"], marker="o")
plt.axhline(0, linestyle="--")
plt.xlabel("Start Hour")
plt.ylabel("Average Gain from Repositioning ($/hr)")
plt.title("Average Value of Dynamic Repositioning by Shift Start Hour")
plt.grid(True)
plt.show()

In [ ]:
# plot distribution of reposition gains

plt.hist(comparison_results["gain_from_reposition"], bins=40)
plt.xlabel("Gain from Repositioning ($/hr)")
plt.ylabel("Number of zone-hour strategies")
plt.title("Distribution of Gains from Dynamic Repositioning")
plt.show()

In [ ]:
# new heatmap

heatmap_data = comparison_results.pivot_table(
    values="reposition_eph",
    index="PULocationID",
    columns="start_hour"
)

plt.figure(figsize=(12,6))

plt.imshow(heatmap_data, aspect="auto")

plt.colorbar(label="Expected Net Earnings ($/hr)")

plt.xlabel("Start Hour")
plt.ylabel("Pickup Zone")

plt.title("Expected Taxi Driver Earnings with Dynamic Repositioning")

plt.show()

In [ ]:
# nyc map visualization

import geopandas as gpd

best_reposition_zone = (
    comparison_results
    .sort_values("reposition_eph", ascending=False)
    .drop_duplicates("PULocationID")
)

zones = gpd.read_file(raw_dir / "taxi_zones" / "taxi_zones.shp")

map_data = zones.merge(
    best_reposition_zone,
    left_on="LocationID",
    right_on="PULocationID"
)

map_data.plot(
    column="reposition_eph",
    cmap="viridis",
    legend=True,
    figsize=(10,10),
    edgecolor="black",
    linewidth=0.2
)

plt.title("Best Expected Taxi Earnings by Start Zone (Dynamic Repositioning)")
plt.axis("off")

plt.show()

In [ ]:
# map where repositioning helps the most

zone_gain = (
    comparison_results
    .groupby("PULocationID")["gain_from_reposition"]
    .mean()
    .reset_index()
)

map_gain = zones.merge(
    zone_gain,
    left_on="LocationID",
    right_on="PULocationID"
)

map_gain.plot(
    column="gain_from_reposition",
    cmap="coolwarm",
    legend=True,
    figsize=(10,10),
    edgecolor="black",
    linewidth=0.2
)

plt.title("Value of Dynamic Repositioning by Zone")
plt.axis("off")

plt.show()